<a href="https://colab.research.google.com/github/bradleyboehmke/uc-bana-7025/blob/main/notebooks/tuesday-your-turn/week-05-lecture.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 5: Data Visualization & EDA 📊

## Today's Mission

1. **Your visualization toolkit** — Pandas, Seaborn, Matplotlib, and Bokeh, and when to reach for each
2. **An EDA framework** — question → structure → distributions → segmentation → story
3. **The semester project** — what a sharp business question looks like

**Follow along with the slides.** Lecture is a tour; this notebook is where the depth lives. Everything we demo in class is here, plus the examples we skip for time.

## Getting Started: Load the Data

We'll use the Complete Journey dataset throughout today. Docs: [bit.ly/completejourney_py](https://cunningjames.github.io/completejourney_py/)

In [ ]:
# You may need to install the package first
# !pip install completejourney-py

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from completejourney_py import get_data

cj = get_data()
transactions  = cj['transactions']
products      = cj['products']
demographics  = cj['demographics']

print("Datasets loaded:", list(cj.keys()))

In [ ]:
# Pre-build the DataFrames we'll use throughout

# Basket-level table — the backbone of the Part 2 worked example.
# One row per basket, with a coupon flag and a per-item spend measure.
baskets = (
    transactions
    .groupby(['household_id', 'basket_id'], as_index=False)
    .agg(basket_spend=('sales_value', 'sum'),
         coupon_disc=('coupon_disc', 'sum'),
         n_lines=('product_id', 'size'))
)
baskets['used_coupon']    = baskets['coupon_disc'] != 0
baskets['spend_per_line'] = baskets['basket_spend'] / baskets['n_lines']

# Joined to demographics (note: not every household has demographics)
baskets_demo = baskets.merge(demographics, on='household_id', how='inner')

# Simpler views used in Part 1
basket_spend = baskets[['household_id', 'basket_id', 'basket_spend']]
basket_demo  = baskets_demo

# Weekly sales
weekly_sales = (
    transactions
    .set_index('transaction_timestamp')['sales_value']
    .resample('W').sum()
    .reset_index()
    .rename(columns={'transaction_timestamp': 'week', 'sales_value': 'total_sales'})
)

# Top 10 departments by revenue
category_totals = (
    transactions
    .merge(products[['product_id', 'department']], on='product_id')
    .groupby('department', as_index=False)['sales_value'].sum()
    .sort_values('sales_value')
    .tail(10)
)

income_order = [
    'Under 15K','15-24K','25-34K','35-49K',
    '50-74K','75-99K','100-124K','125-149K',
    '150-174K','175-199K','200-249K','250K+'
]

# Validated two-series palette used in Part 2 (blue = no coupon, orange = coupon)
C_NO, C_YES = '#2a78d6', '#eb6834'

plt.style.use('default')

print("Ready! baskets, baskets_demo, weekly_sales, category_totals all loaded.")
print(f"{len(baskets):,} baskets  |  {len(baskets_demo):,} with demographics")

---

# Part 1: Your Visualization Toolkit 🧰

Four libraries, four different jobs. The goal today is not to memorize syntax — it's to know which one to reach for.

| Purpose | Tool | Why it wins here |
|---------|------|------------------|
| Quick EDA | **Pandas** `.plot()` | One line, straight off a DataFrame |
| Statistical comparison | **Seaborn** | Handles grouping, ordering, and stats for you |
| Polished reporting | **Matplotlib** | Total control over every element |
| Interactive exploration | **Bokeh** | Renders to the browser — zoom, hover, filter |

These are not four competitors. Pandas and Seaborn are both built **on top of** Matplotlib, so they stack.

## 🐼 Pandas — When Speed Beats Polish

Your first reach when you need a fast answer. Call `.plot()` on any Series or DataFrame and pass `kind=`.

**One variable** → call `.plot()` on a **Series**:
- `kind='hist'` — distribution
- `kind='box'` — spread and outliers
- `kind='line'` — trend (index is the x-axis)

**Two variables** → call `.plot()` on a **DataFrame**, add `x=` and `y=`:
- `kind='scatter'` — relationship between two numeric columns
- `kind='bar'` / `kind='barh'` — category comparison
- `kind='line'` — trend over time

> **Rule of thumb:** if you're typing a third line of formatting, switch libraries.

### Example: Distribution

In [ ]:
basket_spend['basket_spend'].plot(
    kind='hist', bins=40, figsize=(10, 3.5),
    title='Distribution of basket spend', xlabel='Basket spend ($)'
)
plt.tight_layout()
plt.show()

### Example: Category Comparison

In [ ]:
category_totals.plot(
    kind='barh', x='department', y='sales_value',
    figsize=(10, 3.5), legend=False,
    title='Total revenue by department (top 10)', xlabel='Total sales ($)'
)
plt.tight_layout()
plt.show()

### Example: Trend Over Time

In [ ]:
weekly_sales.plot(
    kind='line', x='week', y='total_sales',
    figsize=(10, 3.2), legend=False,
    title='Weekly total sales', ylabel='Total sales ($)'
)
plt.tight_layout()
plt.show()

## 📈 Seaborn — When You're Comparing Groups

Built on Matplotlib, with cleaner syntax for statistical chart types. Shines when you need **grouping, ordering, and distributions with shape**.

Every Seaborn function follows the same pattern:

```python
sns.function(data=df, x='col', y='col', hue='group', order=[...])
```

| Argument | Role |
|----------|------|
| `data=` | the DataFrame — always required |
| `x=` | horizontal axis |
| `y=` | vertical axis (two-variable plots) |
| `hue=` | color by group — adds a third dimension |
| `order=` | control category order — critical for income, size, day-of-week |

| Pandas | Seaborn |
|--------|---------|
| `.plot(kind='hist')` — basic bin counts | `histplot()` — bins + KDE curve in one call |
| `.groupby().mean().plot.bar()` — manual aggregation | `boxplot()` / `barplot()` — handles stats automatically |
| No built-in group ordering | `order=` parameter on every chart |

### Example: Distribution with KDE — `histplot`

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3.5))
sns.histplot(data=basket_demo, x='basket_spend', bins=50, kde=True, ax=ax)
ax.set_xlim(0, 100)
ax.set_xlabel('Basket spend ($)')
ax.set_title('Distribution of basket spend — with density curve')
plt.tight_layout()
plt.show()

### Example: Group Comparison — `boxplot`

This is the one we demo in class. Note `order=` — without it, income brackets come out alphabetically, which is meaningless.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
sns.boxplot(
    data=basket_demo,
    x='income',
    y='basket_spend',
    order=income_order,   # explicit ordering — critical for income brackets
    ax=ax,
    showfliers=False
)
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right', fontsize=8)
ax.set_xlabel('Income bracket')
ax.set_ylabel('Basket spend ($)')
ax.set_title('Basket spend by income bracket')
plt.tight_layout()
plt.show()

### Example: Matrix Pattern — `heatmap`

*We skip this one in lecture for time — run it here.*

In [ ]:
# Build day x hour heatmap data
tx = transactions.copy()
tx['hour'] = tx['transaction_timestamp'].dt.hour
tx['day_of_week'] = tx['transaction_timestamp'].dt.day_name()
day_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
heatmap_data = (
    tx
    .groupby(['day_of_week', 'hour'])
    .size()
    .reset_index(name='trip_count')
    .pivot(index='day_of_week', columns='hour', values='trip_count')
    .reindex(day_order)
)

fig, ax = plt.subplots(figsize=(12, 4))
sns.heatmap(
    heatmap_data.iloc[:, 6:22],   # hours 6am-10pm
    cmap='YlOrRd',
    linewidths=0.3,
    ax=ax,
    cbar_kws={'label': 'Trip count'}
)
ax.set_xlabel('Hour of day')
ax.set_ylabel('')
ax.set_title('Shopping trips by day of week and hour')
plt.tight_layout()
plt.show()

## 🎨 Matplotlib — When the Chart Must Stand Alone

The most widely used Python plotting library, and the **foundation** for Seaborn and Pandas. Use it when the chart needs to stand on its own in a report or presentation.

**This is the library your project report will live on.**

```python
fig, ax = plt.subplots(figsize=(10, 4))
# fig = the overall canvas
# ax  = the plot area — axes, ticks, lines, labels all live here
```

Get handles for both and you can modify anything — including plots that Seaborn or Pandas drew for you.

> In lecture we jump straight from Step 0 to the finished chart. The four steps below are that jump, slowed down. **Work through them** — this is where report-quality figures come from.

### Step 0: Quick Pandas Starting Point

Fast, but it would not survive an executive meeting.

In [ ]:
weekly_sales.plot(kind='line', x='week', y='total_sales', figsize=(10, 3), legend=False)
plt.show()

### Step 1: Move to the Figure / Axes API

Get handles — now you control everything.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(weekly_sales['week'], weekly_sales['total_sales'], linewidth=2)
ax.set_title('Weekly total sales')
ax.set_xlabel('Week')
ax.set_ylabel('Total sales ($)')
plt.tight_layout()
plt.show()

### Step 2: Executive-Ready Formatting

Currency axis, a recessive grid, and no chartjunk spines.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3.2))
ax.plot(weekly_sales['week'], weekly_sales['total_sales'], linewidth=2)
ax.set_title('Weekly total sales — Regork grocery chain', pad=10)
ax.set_xlabel('Week')
ax.yaxis.set_major_formatter(mtick.StrMethodFormatter('${x:,.0f}'))  # currency format
ax.grid(True, alpha=0.3)
for spine in ['top', 'right']:
    ax.spines[spine].set_visible(False)  # clean look
plt.tight_layout()
plt.show()

### Step 3: Highlight the Insight — Annotations

A chart that makes its point without a caption. This is the version from the slides.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3.5))
ax.plot(weekly_sales['week'], weekly_sales['total_sales'], linewidth=2)
ax.set_title('Weekly total sales — note holiday spike in late December', pad=10)
ax.set_xlabel('Week')
ax.yaxis.set_major_formatter(mtick.StrMethodFormatter('${x:,.0f}'))
ax.grid(True, alpha=0.3)
for spine in ['top', 'right']:
    ax.spines[spine].set_visible(False)

# Find and annotate the peak week
peak_idx  = weekly_sales['total_sales'].idxmax()
peak_week = weekly_sales.loc[peak_idx, 'week']
peak_val  = weekly_sales.loc[peak_idx, 'total_sales']

ax.annotate(
    f'Holiday spike\n${peak_val:,.0f}',
    xy=(peak_week, peak_val),
    xytext=(peak_week - pd.Timedelta(weeks=8), peak_val * 0.97),
    arrowprops=dict(arrowstyle='->', lw=1.2),
    fontsize=9
)
plt.tight_layout()
plt.show()

### Matplotlib Is the Foundation

Because Seaborn returns a Matplotlib `ax`, you can mix them — Seaborn draws the chart, Matplotlib polishes it.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))

# --- Seaborn draws the plot ---
sns.boxplot(
    data=basket_demo, x='income', y='basket_spend',
    order=income_order, ax=ax, showfliers=False, color='steelblue'
)

# --- Matplotlib polishes it ---
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right', fontsize=8)
ax.set_xlabel('Income bracket')
ax.set_ylabel('Basket spend ($)')
ax.set_title('Basket spend by income — Seaborn + Matplotlib finishing touches')
ax.yaxis.set_major_formatter(mtick.StrMethodFormatter('${x:,.0f}'))
for spine in ['top', 'right']:
    ax.spines[spine].set_visible(False)
plt.tight_layout()
plt.show()

### Common Gotchas

*Not covered in lecture — keep this handy for the project.*

| Problem | Fix |
|---------|-----|
| Unreadable axis numbers | `ax.yaxis.set_major_formatter(mtick.StrMethodFormatter('${x:,.0f}'))` |
| Cluttered or overlapping labels | `rotation=45`, `ha='right'` on tick labels |
| Overplotting on scatter | `alpha=0.3` on the points |
| Cramped layout | `plt.tight_layout()` or `constrained_layout=True` |
| Inconsistent styles | `plt.style.use()` once at the top of your notebook |

## 🌐 Bokeh — When Others Need to Explore

Bokeh renders to HTML and JavaScript, so charts live in the browser rather than just the notebook. Users can zoom, pan, and hover without writing new code.

Best for dashboards, stakeholder exploration tools, and shareable deliverables.

In [ ]:
from bokeh.plotting import figure, show
from bokeh.models import HoverTool, ColumnDataSource
from bokeh.io import output_notebook

output_notebook()

In [ ]:
source = ColumnDataSource(weekly_sales)

p = figure(
    title='Total Weekly Sales — hover, zoom, and pan to explore',
    x_axis_type='datetime',
    width=750, height=350,
    tools='pan,wheel_zoom,box_zoom,reset,save'
)

p.line('week', 'total_sales', source=source, line_width=2, color='steelblue')

hover = HoverTool(tooltips=[
    ('Week',  '@week{%F}'),
    ('Sales', '@total_sales{$0,0}')
], formatters={'@week': 'datetime'})
p.add_tools(hover)

p.xaxis.axis_label = 'Week'
p.yaxis.axis_label = 'Total Sales ($)'

show(p)

**Try it:** Hover over the spike near the end of the year. Use box zoom to focus on a single month. What can you see here that the static Matplotlib version hides?

---

## 🧑‍💻 Exercise 1 — Run & Compare (5 minutes)

**Every cell below is ready to run — you don't write any code.**

Run them top to bottom. Same underlying question each time — *how does basket spend vary?* — answered with a different tool.

As you go, keep two questions in mind:

1. Which tool got to a usable chart with the **least code**?
2. Which output would you actually put in front of the CEO?

### 1 of 5 — Pandas: a distribution in one line

In [ ]:
basket_spend['basket_spend'].plot(kind='hist', bins=40, figsize=(9, 3),
                                  title='Basket spend distribution')
plt.show()

### 2 of 5 — Pandas: a category comparison

In [ ]:
(basket_demo
 .groupby('income', as_index=False)['basket_spend'].median()
 .set_index('income').reindex(income_order).reset_index()
 .plot(kind='bar', x='income', y='basket_spend', figsize=(9, 3),
       legend=False, title='Median basket spend by income'))
plt.show()

### 3 of 5 — Seaborn: the same comparison, split by a group

One extra argument (`hue=`) adds a whole dimension. Try changing `hue` to `'marital_status'`.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.5))
sns.barplot(data=basket_demo, x='income', y='basket_spend',
            hue='home_ownership', order=income_order,
            estimator='median', errorbar=None, ax=ax)
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right', fontsize=8)
ax.set_title('Median basket spend by income and home ownership')
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

### 4 of 5 — Matplotlib: the executive-ready version

In [ ]:
med = (basket_demo.groupby('income')['basket_spend'].median().reindex(income_order))

fig, ax = plt.subplots(figsize=(9, 3.5))
ax.bar(med.index, med.values, color='#2a78d6')
ax.set_title('Median basket spend rises with income', pad=10)
ax.set_ylabel('Median basket spend')
ax.yaxis.set_major_formatter(mtick.StrMethodFormatter('${x:,.0f}'))
ax.set_xticklabels(med.index, rotation=45, ha='right', fontsize=8)
ax.grid(axis='y', alpha=0.25)
for spine in ['top', 'right']:
    ax.spines[spine].set_visible(False)
plt.tight_layout()
plt.show()

### 5 of 5 — Bokeh: the interactive version

Hover over any bar to read the exact value.

In [ ]:
from bokeh.models import ColumnDataSource, HoverTool

med_df = (basket_demo.groupby('income')['basket_spend'].median()
          .reindex(income_order).reset_index())
src = ColumnDataSource(med_df)

p = figure(x_range=list(med_df['income']), height=320, width=750,
           title='Median basket spend by income — hover to inspect',
           tools='pan,box_zoom,reset,save')
p.vbar(x='income', top='basket_spend', width=0.7, source=src, color='#2a78d6')
p.add_tools(HoverTool(tooltips=[('Income', '@income'), ('Median', '@basket_spend{$0.00}')]))
p.xaxis.major_label_orientation = 0.8
p.yaxis.axis_label = 'Median basket spend ($)'
show(p)

### Your answers

**1. Which tool got to a usable chart with the least code?**

*Write here.*

**2. Which output would you put in front of the CEO — and why?**

*Write here.*

---

## 📚 Where You Actually Learn These

Today was a **tour**. The readings are where you learn to drive.

| Chapter | What it teaches you | Where it shows up in the project |
|---------|---------------------|----------------------------------|
| **13** — Pandas | `.plot()` for fast exploratory charts | How you'll explore your own data |
| **14** — Seaborn / Matplotlib / Bokeh | statistical charts, full control, interactivity | How you'll build report-quality figures |
| **15** — EDA | a complete case study, start to finish | The model for your entire report |

**20 of the project's 75 points ride on your charts.** EDA (10 pts) and Findings (10 pts) are both graded on whether your visuals reveal something non-obvious *and* are clearly narrated.

Nobody produces professional-looking figures on the first attempt. The reps come from working through these chapters.

---

# Part 2: How to Think About EDA 🔍

## You Already Have Most of What EDA Needs

| You learned | Which lets you |
|-------------|----------------|
| **Subsetting & filtering** (Wk 3) | isolate the rows a question is actually about |
| **`groupby` & aggregation** (Wk 4) | compute statistics at *any* level — household, basket, department, week |
| **Joins** (Wk 4) | pull in the context that makes a comparison possible |
| **Visualization** (this week) | see the shape, catch what's wrong, show the finding |

Aggregation tells you **what the numbers are**. Visualization tells you **what they mean**. Together, that is most of an EDA.

What's left is understanding how to **ask a good question of our data**.

## A Framework for EDA

| Step | Question to ask |
|------|-----------------|
| **1. Question** | What do I actually want to know? Is it specific enough to know when I've answered it? |
| **2. Structure** | What shape is the data? What do I need to join? Where are the missings? |
| **3. Distributions** | What does the outcome variable look like overall? Any outliers? |
| **4. Segmentation** | Does the pattern differ across groups? Where does the interesting variation live? |
| **5. Story** | What changed? What surprised me? What should the decision-maker do next? |

## Step 1: Start With a Sharp Question

**Too broad:** *"How do customers shop?"*
No unit of analysis, no metric, no comparison. You could work for a week and never know if you were finished.

**Sharp:** *"Do households that redeem coupons have larger basket spend than those that don't — and does this differ across income brackets?"*

This names four things:

| It names | Which tells us |
|----------|----------------|
| A **unit** — the basket | what one row of our table has to be |
| A **metric** — spend | what to compute, and that skew will matter |
| A **comparison** — coupon vs. not | what the Step 4 chart has to show |
| A **segmentation** — income | what to check next |

We answer this exact question over the next four steps.

## Step 2: Structure — What Do I Actually Have?

Our question compares **baskets**, but `transactions` gives us 1.5 million *item* rows. So the first move is structural: collapse to **one row per basket**, flag which ones used a coupon, then attach household demographics for the income half of the question.

*(We built `baskets` and `baskets_demo` in the setup cell — scroll up to re-read that code.)*

Every one of those steps can quietly lose rows — so count what survives.

In [ ]:
n_baskets   = len(baskets)
n_coupon    = int(baskets['used_coupon'].sum())
pct_coupon  = 100 * baskets['used_coupon'].mean()
n_joined    = len(baskets_demo)
pct_kept    = 100 * n_joined / n_baskets

print(f"Baskets:                    {n_baskets:,}")
print(f"  ... that used a coupon:   {n_coupon:,}  ({pct_coupon:.1f}%)")
print(f"Households:                 {baskets['household_id'].nunique():,}")
print()
print(f"After joining demographics: {n_joined:,} baskets  ({pct_kept:.0f}% kept)")
print(f"  ... households:           {baskets_demo['household_id'].nunique():,}")

> ⚠️ **The join cost us half the data.** Only 801 of 2,469 households have demographics. Any income comparison we make describes *those* households — not all shoppers.
>
> Notice this now, not in your conclusion.

## Step 3: Distributions — Which Summary Can I Trust?

Before comparing any groups, look at the outcome variable alone: **what does a typical basket even look like?**

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3))
sns.histplot(data=baskets, x='basket_spend', bins=60, kde=True, ax=ax)
ax.set_xlim(0, 120)
ax.axvline(baskets['basket_spend'].mean(),   color='crimson', ls='--', lw=1.6,
           label=f"mean  ${baskets['basket_spend'].mean():.2f}")
ax.axvline(baskets['basket_spend'].median(), color='seagreen', ls='-',  lw=1.6,
           label=f"median ${baskets['basket_spend'].median():.2f}")
ax.set_xlabel('Basket spend ($)')
ax.set_title(f"Basket spend is heavily right-skewed (skew = {baskets['basket_spend'].skew():.1f})")
ax.legend()
for spine in ['top', 'right']:
    ax.spines[spine].set_visible(False)
plt.tight_layout()
plt.show()

The mean sits **$12 above** the median — a handful of enormous baskets drag it to the right.

> 📌 **Report the median — and say why you did.**
>
> With skew this strong, the **mean describes a basket almost nobody actually has**. The median is what a typical trip looks like, and it barely moves when a few $300 stock-ups show up.
>
> Put that reasoning in your own report — one sentence is enough: *"We compare medians because basket spend is right-skewed."* It tells your reader the choice was deliberate rather than a default, and defending a judgment call like this is exactly what the EDA portion of the rubric rewards.

## Step 4: Segmentation — Does the Pattern Hold?

In [ ]:
med = baskets.groupby('used_coupon')['basket_spend'].median()

fig, ax = plt.subplots(figsize=(9, 3.2))
bars = ax.bar(['No coupon', 'Used a coupon'], [med[False], med[True]],
              color=[C_NO, C_YES], width=0.5)
for b, v in zip(bars, [med[False], med[True]]):
    ax.text(b.get_x() + b.get_width()/2, v + 1.2, f'${v:,.2f}',
            ha='center', fontsize=11, weight='bold')
ax.set_ylabel('Median basket spend ($)')
ax.set_title('Coupon baskets are 3.3x larger')
ax.set_ylim(0, 62)
for spine in ['top', 'right']:
    ax.spines[spine].set_visible(False)
plt.tight_layout()
plt.show()

### Step 4 (cont.): Does It Differ by Income?

The second half of our Step 1 question. Let's look, rather than assert.

In [ ]:
med_inc = (baskets_demo.groupby(['income', 'used_coupon'])['basket_spend']
           .median().unstack().reindex(income_order))
hh      = baskets_demo.groupby('income')['household_id'].nunique().reindex(income_order)

x, w = np.arange(len(income_order)), 0.38
fig, ax = plt.subplots(figsize=(10, 3.6))

# thin-data band behind the last three brackets
ax.axvspan(x[-3] - 0.5, x[-1] + 0.5, color='#f0eeea', zorder=0)

ax.bar(x - w/2 - 0.01, med_inc[False], w, label='No coupon',     color=C_NO,  zorder=3)
ax.bar(x + w/2 + 0.01, med_inc[True],  w, label='Used a coupon', color=C_YES, zorder=3)

ax.set_xticks(x)
ax.set_xticklabels(income_order, rotation=45, ha='right', fontsize=8)
ax.set_ylabel('Median basket spend ($)')
ax.set_title('Spend rises with income — but the coupon gap holds at every level')
ax.yaxis.set_major_formatter(mtick.StrMethodFormatter('${x:,.0f}'))
ax.grid(axis='y', alpha=0.25, zorder=0)
ax.legend(loc='upper left', frameon=False, fontsize=9)
ax.text(x[-2], ax.get_ylim()[1]*0.90,
        f'only {hh.iloc[-3:].min()}-{hh.iloc[-3:].max()} households',
        ha='center', fontsize=8, color='#6b6b66', style='italic', zorder=4)
for spine in ['top', 'right']:
    ax.spines[spine].set_visible(False)
plt.tight_layout()
plt.show()

# The lift as a ratio, bracket by bracket
lift = ((med_inc[True] / med_inc[False] - 1) * 100).round(0)
print("Coupon lift by income bracket (%):")
print(lift.to_string())

Orange clears blue in **every single bracket**. Both series drift up with income — so the real question is whether the *gap* widens.

It doesn't. As a ratio the lift bounces from **roughly 100% up to 290%** with no clean direction. Richer households spend more per trip, coupon or not — but **the coupon effect itself is not an income story.** The second half of our question resolves to "no," and ruling something out is progress, not failure.

> ⚠️ And notice the shaded brackets: those medians rest on **5–11 households**. That is the Step 2 join loss surfacing exactly where you'd be most tempted to read a trend.

## Step 5: Build the Story — and the "So What" Test

A 3.3x gap is tempting. **Push one question further before you recommend anything.**

In [ ]:
g = baskets.groupby('used_coupon')
lines, per_line = g['n_lines'].median(), g['spend_per_line'].median()

fig, axes = plt.subplots(1, 2, figsize=(10, 2.9))
labels = ['No coupon', 'Coupon']

axes[0].bar(labels, [lines[False], lines[True]], color=[C_NO, C_YES], width=0.5)
axes[0].set_title('Median items per basket')
for i, v in enumerate([lines[False], lines[True]]):
    axes[0].text(i, v + 0.5, f'{v:.0f}', ha='center', weight='bold')
axes[0].set_ylim(0, 23)

axes[1].bar(labels, [per_line[False], per_line[True]], color=[C_NO, C_YES], width=0.5)
axes[1].set_title('Median spend per item')
for i, v in enumerate([per_line[False], per_line[True]]):
    axes[1].text(i, v + 0.08, f'${v:.2f}', ha='center', weight='bold')
axes[1].set_ylim(0, 3.6)

for ax in axes:
    for spine in ['top', 'right']:
        ax.spines[spine].set_visible(False)
plt.tight_layout()
plt.show()

**Spend per item is identical.** Coupon baskets are not *pricier* — they are simply **bigger trips**.

## What We Can — and Cannot — Say

> 📌 **The "So What" Test**
>
> *So what?* Coupon redemption marks the **weekly stock-up trip**, not a change in what people buy.

> ⚠️ **The recommendation we cannot make:** *"Send more coupons to grow basket size."*
>
> Nothing here shows coupons **caused** bigger baskets. Shoppers planning a big trip are the ones who bother to bring a coupon. We found a marker, not a lever.

Run the cell below to see why the Step 3 decision mattered.

In [ ]:
mean_pl   = baskets.groupby('used_coupon')['spend_per_line'].mean()
median_pl = baskets.groupby('used_coupon')['spend_per_line'].median()

print("Spend per item — MEAN (misleading):")
print(f"  no coupon: ${mean_pl[False]:.2f}   coupon: ${mean_pl[True]:.2f}")
print("     -> reads as 'coupon shoppers buy cheaper stuff'\n")
print("Spend per item — MEDIAN (honest):")
print(f"  no coupon: ${median_pl[False]:.2f}   coupon: ${median_pl[True]:.2f}")
print("     -> essentially identical")

**Had we used means, we would have gotten this backwards.** A small number of one-item baskets holding a single expensive product inflate the no-coupon mean. The Step 3 decision is what saved the analysis.

**Next question:** do the *same* households shop differently with and without a coupon? That comparison could support a causal claim. This one can't.

---

## 🤔 Exercise 2 — What's the Next Question? (3 minutes)

We concluded: *coupon baskets are 3.3x larger, but spend per item is identical — coupons mark stock-up trips.*

Your VP of Promotions reads this and asks: **"So should we mail more coupons?"**

Name **3 follow-up analyses** you would run before answering, and what each would tell you.

**Analysis 1:** *Write here — what would it tell you?*

**Analysis 2:** *Write here — what would it tell you?*

**Analysis 3:** *Write here — what would it tell you?*

In [ ]:
# Scratch space — try one of your follow-up ideas here


---

# 🚀 Going Deeper (optional, on your own)

These were cut from lecture for time. They're the fastest way to get better at the part of the project that's actually graded.

## Gallery Hunt — Seaborn

Browse the **[Seaborn example gallery](https://seaborn.pydata.org/examples/)** and find one plot that could reveal something about the Complete Journey data.

- **Plot type I chose:** *Write here*
- **CJ variables I'd use:** *e.g., x='income', y='basket_spend', hue='marital_status'*
- **Insight I hope it reveals:** *Write here*

## Gallery Hunt — Matplotlib

Same exercise with the **[Matplotlib gallery](https://matplotlib.org/stable/gallery/index.html)**.

- **Plot type I chose:** *Write here*
- **CJ variables I'd use:** *Write here*
- **Insight I hope it reveals:** *Write here*

## Chapter 14's Challenge

Chapter 14 ends by asking you to pick a library we did **not** cover — browse [PyViz](https://pyviz.org/) — adapt one gallery example to the Complete Journey data, and come ready to share what surprised you.

That habit (read the docs, run an example, break it, fix it) outlasts any single plotting API.

In [ ]:
# Scratch space — adapt a gallery example to Complete Journey data


---

# 🧾 Quick Reference

## Tool Selection

| Purpose | Tool | When to reach for it |
|---------|------|---------------------|
| Quick EDA | Pandas `.plot()` | Spot-check a distribution or trend in one line |
| Statistical comparison | Seaborn | Compare groups, show distributions with shape |
| Polished reporting | Matplotlib | Final figures for reports and presentations |
| Interactive exploration | Bokeh | Stakeholder tools, hover + zoom, shareable HTML |

## Pandas `.plot()` Cheat Sheet

| Task | Code |
|------|------|
| Distribution | `series.plot(kind='hist', bins=40)` |
| Box plot | `series.plot(kind='box')` |
| Bar chart | `df.plot(kind='bar', x='cat', y='val')` |
| Horizontal bar | `df.plot(kind='barh', x='cat', y='val')` |
| Line chart | `df.plot(kind='line', x='date', y='val')` |
| Scatter | `df.plot(kind='scatter', x='col_a', y='col_b')` |

## Seaborn Cheat Sheet

| Task | Code |
|------|------|
| Distribution + KDE | `sns.histplot(data=df, x='col', kde=True)` |
| Group spread | `sns.boxplot(data=df, x='group', y='col', order=[...])` |
| Relationship | `sns.scatterplot(data=df, x='col_a', y='col_b', hue='group')` |
| Matrix | `sns.heatmap(pivot_df, cmap='YlOrRd')` |
| Group means | `sns.barplot(data=df, x='group', y='col', order=[...])` |

## Matplotlib Polish Cheat Sheet

| Task | Code |
|------|------|
| Currency y-axis | `ax.yaxis.set_major_formatter(mtick.StrMethodFormatter('${x:,.0f}'))` |
| Rotate tick labels | `ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')` |
| Remove top/right spines | `for s in ['top','right']: ax.spines[s].set_visible(False)` |
| Add annotation | `ax.annotate('text', xy=(x, y), xytext=(xt, yt), arrowprops=dict(...))` |
| Grid | `ax.grid(axis='y', alpha=0.25)` |

## EDA Checklist

1. **Question** — can you describe what the answer would look like before writing code?
2. **Structure** — reshape to the right unit, then count what survives every join
3. **Distributions** — check skew before choosing mean vs. median, and say which you chose
4. **Segmentation** — does the pattern hold across groups? Watch thin cells
5. **Story** — apply the "so what" test, and name what you *cannot* conclude

**You're ready for Thursday's lab — project ideation! 🚀**